# Analyze Recorded DJ Set to Produce Chromagram Vectors

This notebook produces chromagram vectors, per time step, for the given recorded show.

Each chromagram vector covers the entire piano keyboard, per note. Stated more technically, each chromagram vector covers each note within the scientific pitch notation range C0 through B8. 

## Import useful libraries

In [ ]:
import librosa
import pandas as pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

In [ ]:
from chromagram_functions import process_octave

## User settings

In [ ]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 300
spark_memory = '70G'

octave_min_inclusive = 0
octave_max_inclusive = 8

# location of the recorded DJ set which we want to analyze
filename_show = '/home/emily/Desktop/projects/dj/song_recognition/data/full-goth-set.mp3'

## Load recorded show and extract harmonic content

In [ ]:
y_show, sr_show = librosa.load(filename_show)
y_show_harmonic, y_show_percussive = librosa.effects.hpss(y_show)

## Define function for extracting per time step features of the recorded DJ set

In [ ]:
def process_show(y_harmonic, sampling_rate, hop_length, octave_min_inclusive = 0, octave_max_inclusive = 8):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now
    results_list = []
    for octave_number in range(octave_min_inclusive, octave_max_inclusive + 1):
        chromagram = process_octave(y_harmonic, octave_number, sr = sampling_rate, hop_length = hop_length)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    return df_all_octaves

## Compute DJ set features per time step

In [ ]:
pdf_show = process_show(
    y_show_harmonic,
    sampling_rate,
    hop_length,
    octave_min_inclusive = octave_min_inclusive,
    octave_max_inclusive = octave_max_inclusive,
)

In [ ]:
pdf_show.head(3)

## QA

In [ ]:
len(pdf_show.index)

In [ ]:
len(pdf_show.dropna().index)

## Identify the column names for the notes

In [ ]:
pitch_columns = [x for x in pdf_show.columns if x != 'id']

## Enumerate the time steps

In [ ]:
pdf_show['time_step'] = pdf_show.index

## Initialize a Spark session

In [ ]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

## Convert Pandas DF to Spark DF and collapse pitch columns to a vector column

In [ ]:
sdf_show = (
    spark
    .createDataFrame(pdf_show)
    .orderBy('time_step')
    .withColumn('array_show', F.array(*pitch_columns))
    .select('time_step', 'array_show')
)

In [ ]:
sdf_show.show(5)

## Save for later

In [ ]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show.write.mode('overwrite').parquet(path_show_output)

## Close Spark session

In [ ]:
spark.stop()